# Module 06 AI Workbook — solution and explanation

> Try the exercise yourself before reading this.

## The adversarial bug

[../adversarial_calorimeter_buggy.py](./adversarial_calorimeter_buggy.py)
accumulates energy with:

```python
cells[cell_ids] += edep      # cell_ids has many repeats (a real shower)
```

NumPy evaluates this as `cells[cell_ids] = cells[cell_ids] + edep`, using the
*original* `cells` on the right and writing each distinct cell once. When 300 hits
land in the same cell, that cell receives only **one** of their energies. Total
energy is not conserved.

This is the exact CPU mirror of a **GPU energy deposition without
`cuda.atomic.add`**: many threads doing `cells[c] += e` into the same cell
read-modify-write concurrently and clobber each other.

## The fix

Use an accumulating scatter-add:

```python
np.add.at(cells, cell_ids, edep)     # CPU
```

or on the GPU:

```python
cuda.atomic.add(cells, c, e)          # Numba CUDA
```

Both are in [calorimeter_solution.py](solutions/calorimeter_solution.py). The GPU path runs
only if `numba` and a CUDA device are present.

## Why the verification matters

An energy map that "looks like a shower profile" passes a casual review while
silently violating energy conservation. The reliable tests are:

1. **`sum(map) == sum(deposits)`** — energy conservation, a physics invariant, and
2. a **per-cell** comparison against the weighted-`bincount` reference.

That is what [../verify_calorimeter.py](./verify_calorimeter.py) checks.

## Mapping to the 5-phase loop

- **PREDICT:** flag `cells[ids] += e` with repeated ids as a scatter race.
- **VERIFY:** use energy conservation plus per-cell comparison, not a plot.
- **DIAGNOSE:** on the GPU, atomics into hot cells serialize; the next step is
  per-block privatized maps merged at the end — the same idea as shared-memory
  histogramming in CUDA.


## Run the worked solution

The cells below build and run the solution. Each should finish with `PASS`.

In [ ]:
!python solutions/calorimeter_solution.py